# OV Value Circuit Analysis

Detailed OV circuit analysis: eigenspectrum, token copying, writing direction,
cross-layer composition, and unembedding alignment.

## Why This Matters

OV circuit value circuit analyzes the value-output projection that determines what information flows through attention. Understanding what each head copies when it attends to a position reveals the head's computational role in the model's circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.ov_value_circuit import (
    ov_eigenspectrum, ov_token_copying_score,
    ov_writing_direction, ov_composition_with_next_layer,
    ov_unembed_alignment,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## OV Eigenspectrum

Singular value decomposition of the OV matrix.

In [ ]:
for layer in range(4):
    print(f'Layer {layer}:')
    for head in range(4):
        result = ov_eigenspectrum(model, layer=layer, head=head)
        print(f"  Head {head}: rank={result['effective_rank']:.2f}, "
              f"spec={result['spectral_norm']:.4f}, "
              f"frob={result['frobenius_norm']:.4f}")
    print()

## Token Copying Score

Does the OV circuit copy token identity through embed → OV → unembed?

In [ ]:
result = ov_token_copying_score(model, layer=0, head=0, token_ids=[1, 10, 20, 50, 80])
copy = 'COPY' if result['is_copy_head'] else 'not copy'
print(f"Mean self-rank: {result['mean_self_rank']:.1f} [{copy}]\n")
for t in result['per_token']:
    c = '✓' if t['is_copying'] else '✗'
    print(f"  Token {t['token']}: self_logit={t['self_logit']:+.4f}, rank={t['self_rank']} [{c}]")

## Writing Direction

What direction does the OV circuit write per source position?

In [ ]:
result = ov_writing_direction(model, tokens, layer=0, head=0)
print(f"Mean output norm: {result['mean_output_norm']:.4f}\n")
for s in result['per_source']:
    bar = '█' * int(abs(s['cosine_with_mean']) * 20)
    print(f"  Src {s['source']} (tok {s['token']:2d}): norm={s['output_norm']:.4f}, "
          f"cos={s['cosine_with_mean']:+.4f} {bar}")

## OV Composition with Next Layer

How does this head's output feed into the next layer's Q and K?

In [ ]:
for layer in range(3):  # skip last layer
    result = ov_composition_with_next_layer(model, layer=layer, head=0)
    print(f"L{layer}H0 → L{result['next_layer']}:")
    for c in result['compositions'][:2]:
        print(f"  → H{c['next_head']}: Q={c['ov_to_q_norm']:.4f}, K={c['ov_to_k_norm']:.4f}")
    print()

## Unembed Alignment

What tokens does the top OV singular vector promote/suppress?

In [ ]:
result = ov_unembed_alignment(model, layer=0, head=0, top_k=5)
print(f"Top SV: {result['top_sv']:.4f}\n")
print('Promoted:')
for t in result['promoted_tokens']:
    print(f"  tok {t['token']}: logit={t['logit']:+.4f}")
print('\nSuppressed:')
for t in result['suppressed_tokens']:
    print(f"  tok {t['token']}: logit={t['logit']:+.4f}")